# AI Bank Pricing Engine Walkthrough

Welcome. This notebook teaches you how to use the AI Bank Pricing Engine from the ground up.

You do **not** need to understand every line of Python to use it. The notebook is designed in two layers:

- **Beginner layer:** read the plain-English notes, run each cell from top to bottom, and focus on the output tables.
- **Technical layer:** inspect the code, change assumptions, rerun scenarios, and review model diagnostics.

The project uses **synthetic data only**. It currently recommends one portfolio-level home loan rate and one portfolio-level term deposit rate. It does **not** do personalized customer pricing.

## How To Use This Notebook

If you are new to Jupyter notebooks, use this simple routine:

1. Click the first code cell.
2. Press **Shift + Enter** to run it.
3. Read the output directly below the cell.
4. Move to the next cell and repeat.
5. Do not edit code until you reach **Section 12: Try Your Own Scenario**.

### Recommended Beginner Path

For your first run, focus on these sections:

- **0-3:** setup and assumptions.
- **4-6:** generate synthetic data and run the baseline optimization.
- **7-9:** understand the recommended pricing decision.
- **11:** check whether the recommendation is sensitive to assumptions.
- **15-17:** learn how to experiment and troubleshoot.

Sections 13 and 14 are useful, but optional for a first pass.

## 0. Before You Start

The notebook assumes the project has already been installed into your Python environment.

From a terminal, run these commands once:

```bash
cd /Users/kevinchang/Documents/ai-bank-pricing-engine
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install -e ".[dev]"
```

Then launch Jupyter:

```bash
python3 -m pip install notebook
python3 -m notebook
```

Open this file:

```text
notebooks/bank_pricing_engine_walkthrough.ipynb
```

### What Success Looks Like

A normal first run should eventually show:

- a **Best pricing decision** table,
- a **Top pricing candidates** table,
- a **Sensitivity analysis** table,
- and response-model diagnostics later in the notebook.

If you get an error, do not panic. The troubleshooting section near the end covers the common ones.

## The Big Picture In One Minute

The engine tests many possible pairs of rates:

```text
home loan rate + term deposit rate
```

For each pair, it estimates:

- how many synthetic home loan customers accept the loan rate,
- how many synthetic term deposit customers take up the deposit rate,
- how much income the bank earns on loans,
- how much credit loss the bank expects,
- how much interest the bank pays on deposits,
- whether deposits are enough to fund lending,
- and whether a funding shortfall penalty applies.

The engine then picks the pair with the highest total economic objective.

## Notebook Workflow
The notebook is now organised as a simple modelling workflow:

| Part | What you do | Why it matters |
|---|---|---|
| **Data steps** | Load assumptions, generate synthetic customers, inspect customer profiles, and build candidate rate grids. | These cells define the portfolio and pricing inputs. |
| **Pricing model steps** | Run the optimiser, inspect the winner, review the objective, check boundaries, and run sensitivity tests. | These cells explain the recommended portfolio-level rates. |
| **Response modelling steps** | Build synthetic response datasets, train simple models, and check calibration plus rate-response direction. | These cells show how learned response models can be evaluated before production use. |
| **Optional inputs** | Try competitor-rate files and decide where to experiment next. | These cells connect the baseline workflow to future scenarios. |

Run the notebook from top to bottom the first time. After that, use the part headings to rerun the section you are exploring.


## Part 1: Data Steps
This part prepares the inputs used by the pricing engine. Nothing has been optimised yet; the goal is to understand the assumptions, synthetic customer data, and candidate rate grids.


### Data Step 1: Imports And Project Setup
Run the next two code cells first. They load Python packages and project functions used later.

**Beginner note:** if these cells run without an error, your notebook can see the project code.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs" / "base.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from sklearn.model_selection import train_test_split

from balance_pricing_engine.config import load_config
from balance_pricing_engine.diagnostics import run_sensitivity_analysis
from balance_pricing_engine.modelling.datasets import (
    DEPOSIT_FEATURE_COLUMNS,
    DEPOSIT_TARGET_COLUMN,
    LOAN_FEATURE_COLUMNS,
    LOAN_TARGET_COLUMN,
    build_deposit_response_dataset,
    build_loan_response_dataset,
)
from balance_pricing_engine.modelling.evaluation import (
    check_rate_response_direction,
    evaluate_binary_classifier,
    evaluate_calibration,
)
from balance_pricing_engine.modelling.models import (
    build_logistic_regression_model,
    fit_response_model,
)
from balance_pricing_engine.rate_grids import build_configured_rate_grids
from balance_pricing_engine.scenarios.runner import run_baseline_diagnostics
from balance_pricing_engine.synthetic.customers import (
    generate_deposit_customers,
    generate_loan_customers,
)
from balance_pricing_engine.utils.random import make_rng

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

### If The Import Cell Fails

Common fixes:

- If you see `ModuleNotFoundError: No module named 'balance_pricing_engine'`, install the project with `python3 -m pip install -e ".[dev]"` from the repo root.
- If you see a missing package like `sklearn`, rerun the install command above.
- If Jupyter is using the wrong Python environment, close Jupyter, activate `.venv`, and launch Jupyter again from the same terminal.

### Data Step 2: Formatting And Plotting Helpers
These helpers keep rates, dollar values, tables, and charts readable throughout the notebook.


In [ ]:
def fmt_rate(value) -> str:
    if value is None or pd.isna(value):
        return "n/a"
    return f"{value:.2%}"


def fmt_bps(value) -> str:
    if value is None or pd.isna(value):
        return "n/a"
    return f"{value:,.0f} bps"


def fmt_money(value) -> str:
    if value is None or pd.isna(value):
        return "n/a"
    return f"${value:,.0f}"


def show_rate_grid(name: str, rates: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "grid": name,
            "rate_decimal": rates,
            "rate_percent": [fmt_rate(rate) for rate in rates],
        }
    )


def format_result_table(df: pd.DataFrame) -> pd.DataFrame:
    formatted = df.copy()
    rate_columns = [column for column in formatted.columns if column.endswith("rate")]
    money_columns = [
        "objective_value",
        "total_objective",
        "loan_interest_income",
        "expected_credit_loss",
        "deposit_interest_cost",
        "wholesale_funding_cost",
        "funding_shortfall_penalty",
        "expected_loan_volume",
        "expected_deposit_volume",
        "funding_shortfall",
    ]
    bps_columns = [column for column in formatted.columns if column.endswith("bps")]

    for column in rate_columns:
        if column in formatted:
            formatted[column] = formatted[column].map(fmt_rate)
    for column in money_columns:
        if column in formatted:
            formatted[column] = formatted[column].map(fmt_money)
    for column in bps_columns:
        if column in formatted:
            formatted[column] = formatted[column].map(fmt_bps)
    return formatted

### Optional Plotting Helpers
These helpers keep the charts readable and make the notebook safe to run even when `matplotlib` is not installed.


In [ ]:
def get_plotting_tools():
    try:
        import matplotlib.pyplot as plt
        from matplotlib.ticker import FuncFormatter, PercentFormatter
    except ImportError:
        print("matplotlib is not installed. To enable charts, run: python3 -m pip install matplotlib")
        return None

    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        pass
    return plt, FuncFormatter, PercentFormatter


def money_axis_formatter(FuncFormatter):
    return FuncFormatter(lambda value, position: f"${value / 1_000_000:.1f}M")


def rate_axis_formatter(PercentFormatter):
    return PercentFormatter(xmax=1.0, decimals=1)


CHART_COLORS = {
    "loan": "#2f6f9f",
    "deposit": "#2f8f5b",
    "positive": "#2f8f5b",
    "negative": "#b85757",
    "highlight": "#d9902f",
    "neutral": "#6b7280",
}


### Data Step 3: Load The Baseline Configuration
The baseline scenario lives in `configs/base.yaml`. Think of this file as the control panel for the model. It contains the market assumptions, simulation sizes, pricing ranges, balance-sheet settings, and random seed.


In [ ]:
config_path = PROJECT_ROOT / "configs" / "base.yaml"
config = load_config(config_path)
config

In [ ]:
config_summary = pd.DataFrame(
    [
        {"section": "simulation", "assumption": "loan customers", "value": config.simulation.n_loan_customers},
        {"section": "simulation", "assumption": "deposit customers", "value": config.simulation.n_deposit_customers},
        {"section": "market", "assumption": "cash rate", "value": fmt_rate(config.market.cash_rate)},
        {"section": "market", "assumption": "wholesale funding spread", "value": fmt_rate(config.market.wholesale_funding_spread)},
        {"section": "market", "assumption": "competitor loan rate", "value": fmt_rate(config.market.competitor_loan_rate)},
        {"section": "market", "assumption": "competitor deposit rate", "value": fmt_rate(config.market.competitor_deposit_rate)},
        {"section": "market", "assumption": "home loan reference rate", "value": fmt_rate(config.market.home_loan_reference_rate)},
        {"section": "market", "assumption": "term deposit reference rate", "value": fmt_rate(config.market.term_deposit_reference_rate)},
        {"section": "balance sheet", "assumption": "required deposit funding ratio", "value": fmt_rate(config.balance_sheet.required_deposit_funding_ratio)},
        {"section": "balance sheet", "assumption": "funding shortfall penalty rate", "value": fmt_rate(config.balance_sheet.funding_shortfall_penalty_rate)},
        {"section": "response", "assumption": "loan rate elasticity", "value": config.response.loan_rate_elasticity},
        {"section": "response", "assumption": "deposit rate elasticity", "value": config.response.deposit_rate_elasticity},
    ]
)
config_summary

### What To Look For In The Assumptions Table

Start with these rows:

- **competitor loan rate:** the market loan rate the synthetic borrowers compare against.
- **competitor deposit rate:** the market deposit rate the synthetic deposit customers compare against.
- **required deposit funding ratio:** the share of lending management wants funded by deposits.
- **loan rate elasticity:** how strongly loan demand reacts when the offered loan rate changes.
- **deposit rate elasticity:** how strongly deposit uptake reacts when the offered deposit rate changes.

You do not need to memorize these. The important thing is that these assumptions drive the recommendation.

### Plain-English Interpretation

The engine is trying to choose one **home loan rate** and one **term deposit rate** for the whole synthetic portfolio.

The objective is:

```text
loan interest income
- expected credit loss
- deposit interest cost
- wholesale funding cost
- funding shortfall penalty
```

The best rate pair is the one with the highest total objective among all tested rate combinations.

### Data Step 4: Generate Synthetic Customers
The project does not use real customer data. It generates synthetic customers so you can test the pricing logic safely.

There are two synthetic portfolios:
- home-loan customers, representing potential lending demand
- term-deposit customers, representing potential funding supply


In [ ]:
rng = make_rng(config.random_seed)
loan_customers = generate_loan_customers(config.simulation.n_loan_customers, rng)
deposit_customers = generate_deposit_customers(config.simulation.n_deposit_customers, rng)

print(f"Loan customers: {len(loan_customers):,}")
print(f"Deposit customers: {len(deposit_customers):,}")

In [ ]:
loan_customers.head()

In [ ]:
deposit_customers.head()

### Beginner Data Dictionary

Useful home-loan columns:

- `loan_amount`: synthetic requested loan balance.
- `credit_score`: synthetic credit score.
- `lvr`: loan-to-value ratio. Higher values usually mean more risk.
- `income`: synthetic borrower income.
- `risk_score`: model-created risk indicator from 0 to 1.
- `expected_loss_rate`: expected annual credit loss rate.

Useful term-deposit columns:

- `deposit_amount`: synthetic deposit balance.
- `relationship_years`: synthetic customer tenure.
- `digital_engagement`: synthetic digital activity score.
- `rate_sensitivity`: how strongly the customer reacts to deposit rates.

In [ ]:
loan_customer_profile = loan_customers[["loan_amount", "credit_score", "lvr", "income", "risk_score", "expected_loss_rate"]].describe().T
deposit_customer_profile = deposit_customers[["deposit_amount", "relationship_years", "digital_engagement", "rate_sensitivity"]].describe().T

loan_customer_profile

In [ ]:
deposit_customer_profile

### Customer Profile Charts
These charts give you a quick visual feel for the synthetic customers before the optimizer prices the portfolio.


In [ ]:
plotting = get_plotting_tools()
if plotting is not None:
    plt, FuncFormatter, PercentFormatter = plotting

    fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
    axes = axes.flatten()
    plot_specs = [
        (loan_customers["loan_amount"], "Requested home loan balances", "Loan amount", CHART_COLORS["loan"], money_axis_formatter(FuncFormatter)),
        (loan_customers["credit_score"], "Credit scores", "Credit score", CHART_COLORS["loan"], None),
        (loan_customers["lvr"], "Loan-to-value ratios", "LVR", CHART_COLORS["loan"], rate_axis_formatter(PercentFormatter)),
        (deposit_customers["deposit_amount"], "Term deposit balances", "Deposit amount", CHART_COLORS["deposit"], money_axis_formatter(FuncFormatter)),
        (deposit_customers["rate_sensitivity"], "Deposit rate sensitivity", "Sensitivity score", CHART_COLORS["deposit"], None),
        (deposit_customers["relationship_years"], "Relationship length", "Years", CHART_COLORS["deposit"], None),
    ]

    for ax, (series, title, xlabel, color, formatter) in zip(axes, plot_specs):
        series.hist(ax=ax, bins=30, color=color, edgecolor="white", alpha=0.9)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Customers")
        if formatter is not None:
            ax.xaxis.set_major_formatter(formatter)

    fig.suptitle("Synthetic portfolio profile", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


### Data Step 5: Build Candidate Rate Grids
The optimiser does an exhaustive grid search. That means it tests every allowed home-loan-rate and term-deposit-rate combination.

Example:

```text
loan rates     = 5.25%, 5.30%, 5.35%, ...
deposit rates  = 3.60%, 3.65%, 3.70%, ...
```

The more grid points you add, the more combinations the optimiser must evaluate.


In [ ]:
loan_grid, deposit_grid = build_configured_rate_grids(config)
rate_grid_table = pd.concat(
    [show_rate_grid("home loan", loan_grid), show_rate_grid("term deposit", deposit_grid)],
    ignore_index=True,
)
rate_grid_table

In [ ]:
print(f"Home loan rates tested: {len(loan_grid)}")
print(f"Term deposit rates tested: {len(deposit_grid)}")
print(f"Total pricing combinations tested before adaptive expansion: {len(loan_grid) * len(deposit_grid):,}")

## Part 2: Pricing Model Steps
This part runs the portfolio-level pricing model and explains its recommendation. The optimiser tests candidate home-loan and term-deposit rates, then ranks each pair by expected risk-adjusted profit after balance-sheet costs and penalties.


### Pricing Model Step 1: Run The Baseline Optimisation
This is the main pricing run. The next cells run the same core workflow as:

```bash
python3 examples/run_baseline.py
```

The output answers:

```text
Which home loan rate and term deposit rate maximise the portfolio objective?
```


In [ ]:
diagnostics = run_baseline_diagnostics(config)
result = diagnostics.best_result

print("Baseline optimization complete")
print(f"Grid rows evaluated in final run: {len(diagnostics.grid_results):,}")
print(f"Adaptive expansion rounds: {diagnostics.expansion_rounds}")

In [ ]:
best_decision = pd.DataFrame(
    [
        {
            "loan_rate": result.loan_rate,
            "deposit_rate": result.deposit_rate,
            "home_loan_spread_to_market_bps": result.home_loan_spread_to_market_bps,
            "term_deposit_spread_to_market_bps": result.term_deposit_spread_to_market_bps,
            "objective_value": result.objective_value,
            "expected_loan_volume": result.expected_loan_volume,
            "expected_deposit_volume": result.expected_deposit_volume,
            "funding_shortfall": result.funding_shortfall,
        }
    ]
)
format_result_table(best_decision)

### How To Read The Best Decision Table

Read this table from left to right:

- **loan_rate:** recommended portfolio-level home loan rate.
- **deposit_rate:** recommended portfolio-level term deposit rate.
- **spreads to market:** how far the recommended rates sit above or below the reference market rates.
- **objective_value:** the model's total economic score. Higher is better under the assumptions.
- **expected_loan_volume:** expected accepted synthetic lending balance.
- **expected_deposit_volume:** expected synthetic deposit balance.
- **funding_shortfall:** how much deposit funding is missing after applying the required funding ratio.

A result with a slightly lower objective may still be interesting if it has a lower funding shortfall or fits management constraints better.

### Pricing Model Step 2: Decompose The Winning Objective
This breaks the winning objective into business components so you can see why the selected rate pair won.


The decomposition explains **why** the winning rate pair won.

Positive amounts add value. Negative amounts are costs or penalties.

In [ ]:
decomposition = diagnostics.winner_decomposition()
objective_decomposition = pd.DataFrame(
    [
        {"component": "loan interest income", "amount": decomposition.loan_interest_income},
        {"component": "expected credit loss", "amount": -decomposition.expected_credit_loss},
        {"component": "deposit interest cost", "amount": -decomposition.deposit_interest_cost},
        {"component": "wholesale funding cost", "amount": -decomposition.wholesale_funding_cost},
        {"component": "funding shortfall penalty", "amount": -decomposition.funding_shortfall_penalty},
        {"component": "total objective", "amount": decomposition.total_objective},
    ]
)
objective_decomposition.assign(amount=objective_decomposition["amount"].map(fmt_money))

In [ ]:
print("Reconciliation check")
print(f"Reported total objective:     {fmt_money(decomposition.total_objective)}")
print(f"Recalculated total objective: {fmt_money(decomposition.reconciled_total_objective)}")

### Visual Objective Breakdown
The bar chart shows which economic components add value and which components reduce value for the winning rate pair.


In [ ]:
plotting = get_plotting_tools()
if plotting is not None:
    plt, FuncFormatter, PercentFormatter = plotting

    chart_data = objective_decomposition[objective_decomposition["component"] != "total objective"].copy()
    colors = [CHART_COLORS["positive"] if amount >= 0 else CHART_COLORS["negative"] for amount in chart_data["amount"]]

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.barh(chart_data["component"], chart_data["amount"], color=colors)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.xaxis.set_major_formatter(money_axis_formatter(FuncFormatter))
    ax.set_title("Winning objective decomposition")
    ax.set_xlabel("Contribution to objective")

    label_offset = chart_data["amount"].abs().max() * 0.02
    for row_number, amount in enumerate(chart_data["amount"]):
        label_x = amount + label_offset if amount >= 0 else amount - label_offset
        alignment = "left" if amount >= 0 else "right"
        ax.text(label_x, row_number, fmt_money(amount), va="center", ha=alignment)

    plt.tight_layout()
    plt.show()


### Pricing Model Step 3: Check Boundary Diagnostics
A boundary warning means the best rate is sitting at the edge of the search range. In plain English, the model may be saying: "the best answer might be outside the range you allowed me to search."


In [ ]:
if diagnostics.boundary_diagnostics.has_warning:
    for warning in diagnostics.boundary_diagnostics.warnings:
        print(f"WARNING: {warning}")
else:
    print("No final boundary warnings.")

if diagnostics.initial_boundary_diagnostics.has_warning:
    print("\nInitial boundary warnings before adaptive expansion:")
    for warning in diagnostics.initial_boundary_diagnostics.warnings:
        print(f"- {warning}")

### Pricing Model Step 4: Compare Top Pricing Candidates
The top candidates help you see whether the winner is clearly better than alternatives. If ranks 1, 2, and 3 are very close, the recommendation may be sensitive to assumptions.


Beginner reading tip:

- If rank 1 is much better than rank 2, the recommendation is more decisive.
- If the top few rows are close, the recommendation depends more heavily on assumptions.
- If a lower-ranked row has much less funding shortfall, it may still be worth business discussion.

In [ ]:
top_candidates = diagnostics.top_results(10)
format_result_table(
    top_candidates[
        [
            "loan_rate",
            "deposit_rate",
            "total_objective",
            "expected_loan_volume",
            "expected_deposit_volume",
            "funding_shortfall",
            "home_loan_spread_to_market_bps",
            "term_deposit_spread_to_market_bps",
        ]
    ]
)

### Pricing Model Step 5: Explore The Grid Results
This section summarises the optimisation surface. The tables show the best achievable objective for each single rate, while the charts show how the two-rate decision behaves together.


In [ ]:
grid_results = diagnostics.grid_results.copy()

best_by_loan_rate = (
    grid_results.sort_values("total_objective", ascending=False)
    .groupby("loan_rate", as_index=False)
    .first()[["loan_rate", "deposit_rate", "total_objective", "expected_loan_volume", "expected_deposit_volume", "funding_shortfall"]]
)

format_result_table(best_by_loan_rate)

In [ ]:
best_by_deposit_rate = (
    grid_results.sort_values("total_objective", ascending=False)
    .groupby("deposit_rate", as_index=False)
    .first()[["deposit_rate", "loan_rate", "total_objective", "expected_loan_volume", "expected_deposit_volume", "funding_shortfall"]]
)

format_result_table(best_by_deposit_rate)

### Grid Search Pictures
These charts show the optimizer surface: where the objective is highest, how each rate behaves on its own, and how close the top candidates are.


In [ ]:
plotting = get_plotting_tools()
if plotting is not None:
    plt, FuncFormatter, PercentFormatter = plotting

    pivot = (
        grid_results.pivot_table(
            index="deposit_rate",
            columns="loan_rate",
            values="total_objective",
            aggfunc="max",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    loan_rates = pivot.columns.to_numpy(dtype=float)
    deposit_rates = pivot.index.to_numpy(dtype=float)
    best_loan_index = int(np.abs(loan_rates - result.loan_rate).argmin())
    best_deposit_index = int(np.abs(deposit_rates - result.deposit_rate).argmin())

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    heatmap = axes[0, 0].imshow(pivot.to_numpy() / 1_000_000, aspect="auto", origin="lower", cmap="viridis")
    axes[0, 0].scatter(best_loan_index, best_deposit_index, marker="*", s=220, color=CHART_COLORS["highlight"], edgecolor="black")
    axes[0, 0].set_title("Objective surface by rate pair")
    axes[0, 0].set_xlabel("Home loan rate")
    axes[0, 0].set_ylabel("Term deposit rate")
    axes[0, 0].set_xticks(range(len(loan_rates)))
    axes[0, 0].set_xticklabels([fmt_rate(rate) for rate in loan_rates], rotation=45, ha="right")
    axes[0, 0].set_yticks(range(len(deposit_rates)))
    axes[0, 0].set_yticklabels([fmt_rate(rate) for rate in deposit_rates])
    fig.colorbar(heatmap, ax=axes[0, 0], label="Objective ($M)")

    axes[0, 1].plot(best_by_loan_rate["loan_rate"], best_by_loan_rate["total_objective"], marker="o", color=CHART_COLORS["loan"])
    axes[0, 1].axvline(result.loan_rate, color=CHART_COLORS["highlight"], linestyle="--", label="winner")
    axes[0, 1].set_title("Best objective by home loan rate")
    axes[0, 1].set_xlabel("Home loan rate")
    axes[0, 1].set_ylabel("Objective")
    axes[0, 1].xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[0, 1].yaxis.set_major_formatter(money_axis_formatter(FuncFormatter))
    axes[0, 1].legend()

    axes[1, 0].plot(best_by_deposit_rate["deposit_rate"], best_by_deposit_rate["total_objective"], marker="o", color=CHART_COLORS["deposit"])
    axes[1, 0].axvline(result.deposit_rate, color=CHART_COLORS["highlight"], linestyle="--", label="winner")
    axes[1, 0].set_title("Best objective by term deposit rate")
    axes[1, 0].set_xlabel("Term deposit rate")
    axes[1, 0].set_ylabel("Objective")
    axes[1, 0].xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 0].yaxis.set_major_formatter(money_axis_formatter(FuncFormatter))
    axes[1, 0].legend()

    top_10 = top_candidates.head(10).copy()
    shortfall = top_10["funding_shortfall"].clip(lower=0)
    if shortfall.max() > 0:
        sizes = 90 + 320 * shortfall / shortfall.max()
    else:
        sizes = 120
    scatter = axes[1, 1].scatter(
        top_10["loan_rate"],
        top_10["deposit_rate"],
        c=top_10["total_objective"] / 1_000_000,
        s=sizes,
        cmap="magma",
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )
    axes[1, 1].scatter(result.loan_rate, result.deposit_rate, marker="*", s=240, color=CHART_COLORS["highlight"], edgecolor="black", label="winner")
    axes[1, 1].set_title("Top 10 pricing candidates")
    axes[1, 1].set_xlabel("Home loan rate")
    axes[1, 1].set_ylabel("Term deposit rate")
    axes[1, 1].xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 1].yaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 1].legend()
    fig.colorbar(scatter, ax=axes[1, 1], label="Objective ($M)")

    plt.tight_layout()
    plt.show()


### Pricing Model Step 6: Run Sensitivity Analysis
Sensitivity analysis asks: **what changes if our assumptions are wrong?**

The notebook reruns the optimiser under alternative assumptions such as:
- weaker or stronger customer rate response
- different wholesale funding cost
- different funding shortfall penalty
- different competitor rates


In [ ]:
sensitivity = run_sensitivity_analysis(config)
format_result_table(sensitivity)

In [ ]:
sensitivity_summary = sensitivity.copy()
sensitivity_summary["loan_rate_change_vs_baseline"] = sensitivity_summary["best_loan_rate"] - sensitivity_summary.loc[0, "best_loan_rate"]
sensitivity_summary["deposit_rate_change_vs_baseline"] = sensitivity_summary["best_deposit_rate"] - sensitivity_summary.loc[0, "best_deposit_rate"]
sensitivity_summary["objective_change_vs_baseline"] = sensitivity_summary["objective_value"] - sensitivity_summary.loc[0, "objective_value"]

formatted_sensitivity_summary = sensitivity_summary[
    [
        "scenario_name",
        "loan_rate_change_vs_baseline",
        "deposit_rate_change_vs_baseline",
        "objective_change_vs_baseline",
        "funding_shortfall",
        "boundary_warning_flag",
    ]
].copy()

for column in ["loan_rate_change_vs_baseline", "deposit_rate_change_vs_baseline"]:
    formatted_sensitivity_summary[column] = formatted_sensitivity_summary[column].map(lambda x: f"{x * 10_000:+.0f} bps")
formatted_sensitivity_summary["objective_change_vs_baseline"] = formatted_sensitivity_summary["objective_change_vs_baseline"].map(fmt_money)
formatted_sensitivity_summary["funding_shortfall"] = formatted_sensitivity_summary["funding_shortfall"].map(fmt_money)
formatted_sensitivity_summary

### Sensitivity Pictures
These charts make it easier to see which assumptions move the recommendation and which scenarios mainly move profit.


In [ ]:
plotting = get_plotting_tools()
if plotting is not None:
    plt, FuncFormatter, PercentFormatter = plotting

    chart_data = sensitivity_summary.copy()
    y_positions = np.arange(len(chart_data))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    objective_colors = [CHART_COLORS["positive"] if value >= 0 else CHART_COLORS["negative"] for value in chart_data["objective_change_vs_baseline"]]
    axes[0].barh(chart_data["scenario_name"], chart_data["objective_change_vs_baseline"], color=objective_colors)
    axes[0].axvline(0, color="#333333", linewidth=1)
    axes[0].xaxis.set_major_formatter(money_axis_formatter(FuncFormatter))
    axes[0].set_title("Objective change vs baseline")
    axes[0].set_xlabel("Change in objective")

    axes[1].barh(y_positions - 0.18, chart_data["loan_rate_change_vs_baseline"] * 10_000, height=0.35, label="home loan", color=CHART_COLORS["loan"])
    axes[1].barh(y_positions + 0.18, chart_data["deposit_rate_change_vs_baseline"] * 10_000, height=0.35, label="term deposit", color=CHART_COLORS["deposit"])
    axes[1].axvline(0, color="#333333", linewidth=1)
    axes[1].set_yticks(y_positions)
    axes[1].set_yticklabels(chart_data["scenario_name"])
    axes[1].set_title("Recommended rate movement")
    axes[1].set_xlabel("Change vs baseline (bps)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


### Pricing Model Step 7: Try Your Own Scenario
This is the first section where beginners can safely experiment. The example below changes the config **in memory only**. It does not edit `configs/base.yaml`.


In [ ]:
scenario_config = config.model_copy(
    update={
        "balance_sheet": config.balance_sheet.model_copy(
            update={
                "funding_shortfall_penalty_rate": config.balance_sheet.funding_shortfall_penalty_rate * 1.5
            }
        )
    }
)

scenario_diagnostics = run_baseline_diagnostics(scenario_config)
scenario_result = scenario_diagnostics.best_result

comparison = pd.DataFrame(
    [
        {
            "scenario": "baseline",
            "loan_rate": result.loan_rate,
            "deposit_rate": result.deposit_rate,
            "objective_value": result.objective_value,
            "expected_loan_volume": result.expected_loan_volume,
            "expected_deposit_volume": result.expected_deposit_volume,
            "funding_shortfall": result.funding_shortfall,
        },
        {
            "scenario": "higher funding penalty",
            "loan_rate": scenario_result.loan_rate,
            "deposit_rate": scenario_result.deposit_rate,
            "objective_value": scenario_result.objective_value,
            "expected_loan_volume": scenario_result.expected_loan_volume,
            "expected_deposit_volume": scenario_result.expected_deposit_volume,
            "funding_shortfall": scenario_result.funding_shortfall,
        },
    ]
)
format_result_table(comparison)

## Part 3: Response Modelling Steps
This part switches from the pricing optimiser to learned response models. It creates synthetic observed outcomes, trains simple classifiers, and checks whether the learned models behave sensibly as offered rates move.


### Response Modelling Step 1: Train Synthetic Response Models
This section is optional for a first beginner run. The optimiser itself uses known synthetic response functions. This section trains simple models on synthetic observed outcomes to demonstrate a modelling workflow:

```text
synthetic customer data + offered rate -> observed accept / not accept
```

The goal is not to create a production model. The goal is to show the checks a response model should pass before you trust it.


In [ ]:
rng = make_rng(config.random_seed)
loan_model_customers = generate_loan_customers(config.simulation.n_loan_customers, rng)
deposit_model_customers = generate_deposit_customers(config.simulation.n_deposit_customers, rng)
loan_grid, deposit_grid = build_configured_rate_grids(config)

loan_dataset = build_loan_response_dataset(
    loan_model_customers,
    loan_grid,
    config.market.competitor_loan_rate,
    config.response.loan_rate_elasticity,
    config.response.loan_competitor_gap_elasticity,
    rng,
    n_samples=config.simulation.n_loan_customers * 3,
)

deposit_dataset = build_deposit_response_dataset(
    deposit_model_customers,
    deposit_grid,
    config.market.competitor_deposit_rate,
    config.response.deposit_rate_elasticity,
    config.response.deposit_competitor_gap_elasticity,
    rng,
    n_samples=config.simulation.n_deposit_customers * 3,
)

print(f"Loan response rows: {len(loan_dataset):,}")
print(f"Deposit response rows: {len(deposit_dataset):,}")

In [ ]:
loan_dataset.head()

In [ ]:
deposit_dataset.head()

In [ ]:
def train_response_model_report(
    title: str,
    dataset: pd.DataFrame,
    feature_columns: list[str],
    target_column: str,
    offered_rates: list[float],
    competitor_rate: float,
    expected_direction: str,
    random_seed: int,
) -> dict[str, object]:
    train, test = train_test_split(
        dataset,
        test_size=0.30,
        random_state=random_seed,
        stratify=dataset[target_column],
    )
    model = fit_response_model(
        build_logistic_regression_model(random_seed),
        train,
        feature_columns,
        target_column,
    )
    metrics = evaluate_binary_classifier(model, test, feature_columns, target_column)
    calibration = evaluate_calibration(model, test, feature_columns, target_column, n_bins=5)
    response_check = check_rate_response_direction(
        model,
        test.head(1_000),
        feature_columns,
        offered_rates,
        competitor_rate,
        expected_direction,
    )
    return {
        "title": title,
        "model": model,
        "metrics": metrics,
        "calibration": calibration,
        "response_check": response_check,
        "train_rows": len(train),
        "test_rows": len(test),
    }

loan_model_report = train_response_model_report(
    title="Home loan acceptance model",
    dataset=loan_dataset,
    feature_columns=LOAN_FEATURE_COLUMNS,
    target_column=LOAN_TARGET_COLUMN,
    offered_rates=[float(rate) for rate in loan_grid],
    competitor_rate=config.market.competitor_loan_rate,
    expected_direction="decreasing",
    random_seed=config.random_seed,
)

deposit_model_report = train_response_model_report(
    title="Term deposit uptake model",
    dataset=deposit_dataset,
    feature_columns=DEPOSIT_FEATURE_COLUMNS,
    target_column=DEPOSIT_TARGET_COLUMN,
    offered_rates=[float(rate) for rate in deposit_grid],
    competitor_rate=config.market.competitor_deposit_rate,
    expected_direction="increasing",
    random_seed=config.random_seed,
)

In [ ]:
model_metrics = pd.DataFrame(
    [
        {"model": loan_model_report["title"], **loan_model_report["metrics"]},
        {"model": deposit_model_report["title"], **deposit_model_report["metrics"]},
    ]
)
model_metrics

### Model Metric Interpretation

- **AUC**: ranking quality. Higher is better; `0.5` is no better than random ranking.
- **Log loss**: probability accuracy with a strong penalty for confident wrong predictions. Lower is better.
- **Brier score**: squared probability error. Lower is better.

These are diagnostics for the synthetic response-model layer, not production model approval metrics.

In [ ]:
print(loan_model_report["title"])
display(loan_model_report["calibration"])
print(deposit_model_report["title"])
display(deposit_model_report["calibration"])

In [ ]:
loan_response_check = loan_model_report["response_check"]
deposit_response_check = deposit_model_report["response_check"]

print(f"Loan response direction sane: {loan_response_check.is_directionally_sane} | violations: {loan_response_check.violations}")
display(loan_response_check.rate_response.assign(offered_rate=loan_response_check.rate_response["offered_rate"].map(fmt_rate)))

print(f"Deposit response direction sane: {deposit_response_check.is_directionally_sane} | violations: {deposit_response_check.violations}")
display(deposit_response_check.rate_response.assign(offered_rate=deposit_response_check.rate_response["offered_rate"].map(fmt_rate)))

### Response Model Pictures
The calibration plots compare predicted probabilities with observed synthetic outcomes. The response curves show whether predicted customer behaviour moves in the expected direction as rates change.


In [ ]:
plotting = get_plotting_tools()
if plotting is not None:
    plt, FuncFormatter, PercentFormatter = plotting

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    def plot_calibration(ax, report, title):
        calibration = report["calibration"]
        marker_sizes = 80 + 220 * calibration["count"] / calibration["count"].max()
        ax.plot([0, 1], [0, 1], color=CHART_COLORS["neutral"], linestyle="--", label="perfect calibration")
        ax.scatter(
            calibration["mean_predicted_probability"],
            calibration["observed_rate"],
            s=marker_sizes,
            color=CHART_COLORS["highlight"],
            edgecolor="black",
            alpha=0.85,
        )
        ax.set_title(title)
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed response rate")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
        ax.yaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
        ax.legend()

    plot_calibration(axes[0, 0], loan_model_report, "Home loan model calibration")
    plot_calibration(axes[0, 1], deposit_model_report, "Term deposit model calibration")

    axes[1, 0].plot(
        loan_response_check.rate_response["offered_rate"],
        loan_response_check.rate_response["mean_predicted_probability"],
        marker="o",
        color=CHART_COLORS["loan"],
    )
    axes[1, 0].axvline(config.market.competitor_loan_rate, color=CHART_COLORS["neutral"], linestyle="--", label="competitor rate")
    axes[1, 0].set_title("Home loan acceptance response")
    axes[1, 0].set_xlabel("Offered home loan rate")
    axes[1, 0].set_ylabel("Mean predicted acceptance")
    axes[1, 0].xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 0].yaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 0].legend()

    axes[1, 1].plot(
        deposit_response_check.rate_response["offered_rate"],
        deposit_response_check.rate_response["mean_predicted_probability"],
        marker="o",
        color=CHART_COLORS["deposit"],
    )
    axes[1, 1].axvline(config.market.competitor_deposit_rate, color=CHART_COLORS["neutral"], linestyle="--", label="competitor rate")
    axes[1, 1].set_title("Term deposit uptake response")
    axes[1, 1].set_xlabel("Offered term deposit rate")
    axes[1, 1].set_ylabel("Mean predicted uptake")
    axes[1, 1].xaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 1].yaxis.set_major_formatter(rate_axis_formatter(PercentFormatter))
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()


## Part 4: Optional Inputs And Next Steps
This final part covers optional competitor-rate inputs, experiment ideas, glossary terms, troubleshooting, and command-line equivalents.


### Optional Step 1: Run With Competitor Rate Inputs
This section is optional. The baseline notebook already has competitor assumptions in `configs/base.yaml`. This section only matters if you have generated a competitor-rate file under `data/processed/competitor_rates/latest.json`.


In [ ]:
competitor_file = PROJECT_ROOT / "data" / "processed" / "competitor_rates" / "latest.json"
print(f"Competitor file exists: {competitor_file.exists()}")
print(competitor_file)

In [ ]:
if competitor_file.exists():
    from balance_pricing_engine.competitor_rates.apply import apply_competitor_rates_to_config, load_and_resolve_competitor_rates
    from balance_pricing_engine.competitor_rates.models import CompetitorRatesConfig

    competitor_config = config.competitor_rates or CompetitorRatesConfig()
    competitor_config = competitor_config.model_copy(
        update={"enabled": True, "source_file": str(competitor_file)}
    )
    competitor_enabled_config = config.model_copy(update={"competitor_rates": competitor_config})

    try:
        resolved = load_and_resolve_competitor_rates(competitor_config)
        applied_config = apply_competitor_rates_to_config(competitor_enabled_config)
        competitor_diagnostics = run_baseline_diagnostics(applied_config)
        competitor_result = competitor_diagnostics.best_result

        competitor_summary = pd.DataFrame(
            [
                {
                    "mortgage_competitor_reference": resolved.mortgage_reference_rate,
                    "term_deposit_competitor_reference": resolved.term_deposit_reference_rate,
                    "loan_rate": competitor_result.loan_rate,
                    "deposit_rate": competitor_result.deposit_rate,
                    "objective_value": competitor_result.objective_value,
                    "funding_shortfall": competitor_result.funding_shortfall,
                }
            ]
        )
        display(format_result_table(competitor_summary))

        if resolved.validation_warnings:
            print("Validation warnings:")
            for warning in resolved.validation_warnings:
                print(f"- {warning}")
    except (FileNotFoundError, ValueError) as exc:
        print("Competitor-rate file could not be applied to this strategy.")
        print(f"Reason: {exc}")
        print("You can still use the baseline synthetic scenario above.")
else:
    print("No competitor-rate file found. Run the competitor-rate ingestion workflow first if you want this section.")

### Optional Step 2: Where To Experiment Next
The safest beginner workflow is:
1. Change one assumption at a time.
2. Rerun the baseline optimisation.
3. Compare the recommendation and objective.
4. Ask whether the direction makes business sense.


### Reference: Beginner Glossary
- **Portfolio-level pricing:** one rate for the whole synthetic portfolio, not a personalized rate for each customer.
- **Home loan rate:** the interest rate charged on the lending product.
- **Term deposit rate:** the interest rate paid on the funding product.
- **Rate grid:** the list of candidate rates the optimiser tests.
- **Objective:** the value the optimiser is trying to maximise.
- **Expected loan volume:** loan balances expected to be accepted at a given home loan rate.
- **Expected deposit volume:** deposit balances expected to be placed at a given term deposit rate.
- **Funding shortfall:** the amount of required deposit funding not covered by expected deposits.
- **Boundary warning:** a signal that the chosen rate is at the edge of the tested range.
- **Sensitivity analysis:** rerunning the model under changed assumptions to see how stable the answer is.
- **AUC:** a model metric for ranking quality. Higher is better.
- **Calibration:** whether predicted probabilities line up with observed outcome rates.


### Reference: Troubleshooting
#### `ModuleNotFoundError: No module named 'balance_pricing_engine'`

Run this from the repo root:

```bash
python3 -m pip install -e ".[dev]"
```

Then restart the notebook kernel and rerun from the top.

#### `matplotlib is not installed`

Install the project with development dependencies in the same Python environment used by the notebook kernel:

```bash
python3 -m pip install -e ".[dev]"
```

If you use the included Jupyter kernel, select **AI Bank Pricing Engine (Python 3.12)**, restart the kernel, and rerun the plotting cells.

#### Results change after editing assumptions

That is expected. The project uses synthetic data and a fixed random seed. If you change the seed or assumptions, the optimal pricing decision may move.

#### Boundary warnings appear

Boundary warnings do not mean the code failed. They mean the best tested answer sits at the edge of the tested range. Consider widening the relevant rate grid or reviewing the elasticity and penalty assumptions.


### Reference: Command-Line Equivalents
Everything important in this notebook maps back to these scripts:

```bash
python3 examples/run_baseline.py
python3 examples/train_response_models.py
python3 examples/run_with_competitor_rates.py
```

Use the notebook when you want explanation and charts. Use the command line when you want a fast repeatable run.


## Final Beginner Checklist

After your first complete run, you should be able to answer these questions:

1. What home loan rate did the model recommend?
2. What term deposit rate did the model recommend?
3. Was there a funding shortfall?
4. Were there any final boundary warnings?
5. Which sensitivity cases changed the recommendation the most?
6. Which assumption would you like to test next?

If you can answer those, you are using the notebook correctly.